# Unsloth Deep Dive #01 — Internals

Unsloth nasıl 2× hızlı + %70 az VRAM yapıyor? Bu notebook **kaputu açıp içine bakıyor**.

## Bu Notebook Sana Ne Verir?

1. **Mimarinin haritası** — paket nasıl organize edilmiş
2. **Patching mekanizması** — `import unsloth` neden en başta olmalı
3. **Triton kernel'lar** — hangileri, ne yapıyorlar, neden hızlı
4. **Manuel backprop** — neden Unsloth `autograd`'i bypass ediyor
5. **Model registry** — hangi model nasıl tespit ediliyor
6. **Memory tasarrufu** — fused operasyonların matematiği

Bu bilgiler debug ettiğin zamanlarda hayat kurtarır. Hata mesajını okurken "a, bu patch sırasında oluyor" demek paha biçilmez.

## 1. Paket Organizasyonu — Yüksek Seviye Harita

Unsloth iki paketten oluşur:

```
unsloth/                          ← yüksek seviye API + model wrapper'ları
  ├── __init__.py                 ← patching trigger
  ├── models/                     ← model-specific kod
  │   ├── loader.py               ← FastModel, FastLanguageModel, FastVisionModel
  │   ├── llama.py                ← FastLlamaModel (legacy)
  │   ├── qwen3.py, qwen3_moe.py  ← model-specific patches
  │   ├── mistral.py, gemma.py
  │   ├── vision.py               ← VLM model handling
  │   └── _utils.py
  ├── kernels/                    ← Triton kernel'lar (asıl iş burada)
  │   ├── cross_entropy_loss.py   ← 463 satır
  │   ├── fast_lora.py            ← 730 satır (LoRA forward+backward)
  │   ├── rope_embedding.py       ← 465 satır
  │   ├── rms_layernorm.py        ← 339 satır
  │   ├── swiglu.py               ← 143 satır
  │   ├── geglu.py                ← 290 satır
  │   ├── layernorm.py            ← 227 satır
  │   ├── flex_attention.py       ← 187 satır
  │   ├── fp8.py                  ← 624 satır (FP8 quantization)
  │   └── moe/                    ← MoE kernel'ları (yeni 2026)
  ├── trainer.py                  ← UnslothTrainer (CPT için)
  ├── save.py                     ← GGUF/merged save
  └── tokenizer_utils.py

unsloth_zoo/                      ← shared utility functions
  ├── compiler.py                 ← model compile/patch logic
  ├── patch_torch_functions.py    ← PyTorch func override'ları
  ├── patching_utils.py           ← genel patch helpers
  ├── gradient_checkpointing.py   ← Unsloth-spesifik checkpointing
  ├── fused_losses/               ← fused loss kernel'lar
  ├── flex_attention/             ← flex attention impl
  ├── peft_utils.py               ← PEFT (LoRA) entegrasyonu
  ├── temporary_patches/          ← geçici fix'ler
  ├── rl_replacements.py          ← RL trainer patches
  └── vllm_*.py                   ← vLLM entegrasyonu
```

**Toplam Triton kernel kodu:** ~4600 satır. Bu "sadece bir kütüphane" değil — derinlemesine optimize edilmiş tam bir GPU programı.

In [ ]:
# Paket yapısını gözle
import unsloth, os

unsloth_dir = os.path.dirname(unsloth.__file__)
print(f'Unsloth dir: {unsloth_dir}\n')

# Kernel dosyalarını listele
kernel_dir = os.path.join(unsloth_dir, 'kernels')
if os.path.exists(kernel_dir):
    print('=== Triton Kernels ===')
    for f in sorted(os.listdir(kernel_dir)):
        if f.endswith('.py'):
            path = os.path.join(kernel_dir, f)
            with open(path) as fh:
                lines = sum(1 for _ in fh)
            print(f'  {f:30s} {lines:5d} lines')

## 2. Patching Mekanizması — `import unsloth` Neden Başta?

### Sorun

Unsloth, `transformers` kütüphanesinin **fonksiyon ve sınıflarını runtime'da değiştiriyor**. Örneğin:

- `LlamaRotaryEmbedding.forward` → kendi Triton-tabanlı versiyonu
- `LlamaMLP.forward` → fused SwiGLU kernel
- `LlamaRMSNorm.forward` → fused RMSNorm kernel
- Cross-entropy loss → fused CE kernel

Buna **monkey patching** denir. Sorun: monkey patching **import edildikten sonra** yapılan referansları değiştiremez.

```python
# YANLIŞ sıra
from transformers import LlamaForCausalLM   # 1. transformers yüklendi
from transformers.models.llama.modeling_llama import LlamaRMSNorm
import unsloth                                # 2. patch çağrıldı
                                               # → LlamaRMSNorm patch'lendi MAMA
                                               # → biz zaten yukarıda eski referansı aldık!
model = LlamaForCausalLM.from_pretrained(...)  # 3. eski (patch'lenmemiş) versiyonu kullanır
```

### Doğru Sıra

```python
import unsloth                  # 1. ÖNCE — patch'leri kaydet
from transformers import ...     # 2. transformers import edildiğinde patch'lenir
model = ...                      # 3. patched versiyonu kullanır
```

Unsloth ayrıca import-time'da `transformers`, `peft`, `bitsandbytes` modüllerinin yüklenmiş olup olmadığını kontrol eder. Önceden yüklenmişse **uyarır**:

```
Unsloth: Should be imported before trl, transformers, peft. 
Your code may run slower or encounter memory issues.
```

### Patch Türleri

Unsloth iki seviyede patch yapar:

**1. Import-time patches** (`unsloth_zoo.patching_utils`)
- `transformers.modeling_utils.PreTrainedModel.from_pretrained` override
- `peft.tuners.lora.LoraLayer` patch
- `bitsandbytes` low-level CUDA call'larında değişiklik

**2. Load-time patches** (`unsloth/models/*.py`)
- Model yüklendikten sonra, model class'ına spesifik patch
- Örnek: `LlamaForCausalLM` yüklendiyse `models/llama.py` aktif olur
- `LlamaDecoderLayer.forward` Triton versiyonuyla değiştirilir

In [ ]:
# Patch'lenmiş fonksiyonların adresleri değişmiş mi?
import unsloth
from transformers.models.llama.modeling_llama import LlamaRMSNorm
from transformers.models.qwen3.modeling_qwen3 import Qwen3RMSNorm

# Patch'lenmiş forward'lar farklı dosyada (unsloth/kernels/*) gösterilir
import inspect
for cls in [LlamaRMSNorm, Qwen3RMSNorm]:
    fwd = cls.forward
    src_file = inspect.getsourcefile(fwd) or '?'
    is_patched = 'unsloth' in src_file.lower()
    print(f'{cls.__name__:20s} forward → {src_file}  (patched: {is_patched})')

## 3. Triton Kernel'lar — Asıl İş

Triton, OpenAI'nin **GPU programlama dili**. CUDA C yerine Python-tarzı yazıyorsun, compiler optimize CUDA üretiyor. Unsloth'un performans gizemi: kritik operasyonları Triton'a yazıp PyTorch eager mode'unu bypass etmek.

### En Önemli Kernel'lar

**`fast_lora.py` (730 satır) — LoRA forward + backward**

Standart PEFT'te LoRA şöyle:
```
y = x @ W_base + (x @ A @ B) * scale
```
Bu **3 ayrı matmul**. Each requires read x from VRAM. Unsloth tek fused kernel'da yapıyor.

**`cross_entropy_loss.py` (463 satır) — Fused CE**

Standart cross-entropy:
```
logits = model(x)            # [batch, seq, vocab] — VRAM'de tutuluyor
log_probs = log_softmax(logits)
loss = -log_probs[targets]
```
Vocab ~150K olunca `logits` tensörü çok büyük (`batch * seq * 150K * 2 byte` bf16'da). 2K context ile 600 MB! 

Unsloth fused CE: `logits` materialize edilmez, `log_softmax + nll` tek kernel'da yapılır. **VRAM kazancı: ~%50**.

**`rms_layernorm.py` (339 satır) — Fused RMSNorm**

Llama, Qwen, Mistral hepsi RMSNorm kullanır. PyTorch'un standart implementasyonu 4-5 ayrı kernel:
```
var = x.pow(2).mean(-1)      # kernel 1: square + reduce
rstd = 1 / sqrt(var + eps)    # kernel 2: rsqrt
y = x * rstd                  # kernel 3: multiply
y = y * weight                # kernel 4: scale
```
Unsloth tek kernel — register'da hesaplar, hiç ara tensör yazmaz.

**`rope_embedding.py` (465 satır) — Fused RoPE**

Rotary Position Embedding (RoPE) trig fonksiyonlarını her layer'da hesaplar. Unsloth precompute + fused apply ile bu maliyeti düşürür.

**`swiglu.py` (143 satır) — Fused SwiGLU**

FFN'in modern hali: `silu(gate(x)) * up(x) → down(...)`. PyTorch'ta 3 matmul + activation + multiply. Unsloth fused.

### MoE Kernel'lar (yeni 2026)

`kernels/moe/` altında ayrı modül. Sparse routing'i kernel seviyesinde handle eden grouped GEMM. Bu rehberin Notebook 05'inde detaylı.

### Fused Loss'lar (`unsloth_zoo/fused_losses/`)

DPO, ORPO, KTO, GRPO için ayrı fused loss kernel'lar. Notebook 06-07'de.

In [ ]:
# Bir kernel'ın iç yapısına bak
import os

kernel_path = os.path.join(
    os.path.dirname(__import__('unsloth').__file__),
    'kernels', 'rms_layernorm.py'
)

with open(kernel_path) as f:
    content = f.read()

# @triton.jit decorator'larını bul
import re
kernels = re.findall(r'@triton\.(jit|autotune)[\s\S]*?def (\w+)', content)
print(f'rms_layernorm.py içindeki Triton kernel\'lar:')
for kind, name in kernels:
    print(f'  @{kind}: {name}')

# İlk kernel'ın imzasını göster
if kernels:
    first_name = kernels[0][1]
    pattern = re.compile(rf'def {first_name}\(([\s\S]*?)\):')
    m = pattern.search(content)
    if m:
        print(f'\n{first_name} signature:\n{m.group(0)}')

## 4. Manuel Backprop — `autograd`'i Bypass Etmek

PyTorch'un autograd'i forward graph'ı kaydedip backward'da computational graph üzerinden geriye yürür. Bu **rahattır ama yavaştır**:
1. Her forward operasyonda graph node oluşturulur
2. Intermediate tensor'lar VRAM'de tutulur (backward için lazım)
3. Backward generic — operation-spesifik optimizasyon yapamaz

Unsloth manuel backprop yazıyor — `torch.autograd.Function` kullanarak:

```python
class FastRMSNorm(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, eps):
        # 1. Triton kernel ile RMSNorm hesapla
        y = _rms_layernorm_forward_kernel(x, weight, eps)
        # 2. Backward için sadece gerekli tensor'ları kaydet
        ctx.save_for_backward(x, weight)
        ctx.eps = eps
        return y
    
    @staticmethod
    def backward(ctx, grad_output):
        # 3. Manuel backward — fused Triton kernel
        x, weight = ctx.saved_tensors
        grad_x, grad_w = _rms_layernorm_backward_kernel(
            grad_output, x, weight, ctx.eps
        )
        return grad_x, grad_w, None
```

**Avantajlar:**
1. Backward kendi kernel'ında — fused, optimize edilmiş
2. Sadece gerçekten gerekli tensor'lar kaydedilir (autograd hepsini kaydeder)
3. Operation-spesifik matematik kullanılabilir (örn: `x` simetrik ise transpose etme)

### Sayısal Doğruluk

Unsloth iddiası: **0% accuracy loss**. Çünkü:
- Operation matematik olarak aynı
- bf16 ↔ fp32 dönüşümleri standart bound'larda
- Manuel backprop'un türevleri matematiksel olarak doğrulanmış

Fark sadece **execution path**'te. Sonuç tensor'ları byte-byte aynı.

In [ ]:
# Unsloth'un autograd.Function örneklerini incele
import os, re

kernels_dir = os.path.join(os.path.dirname(__import__('unsloth').__file__), 'kernels')

for f in os.listdir(kernels_dir):
    if not f.endswith('.py') or f.startswith('_'):
        continue
    path = os.path.join(kernels_dir, f)
    with open(path) as fh:
        content = fh.read()
    # autograd.Function class'larını bul
    classes = re.findall(r'class (\w+)\(.*?(?:autograd\.Function|Function).*?\):', content)
    if classes:
        print(f'{f}:')
        for c in classes:
            print(f'  {c}')

## 5. Model Registry — Tespit Mekanizması

Unsloth bir model yüklerken **hangi tip olduğunu otomatik tespit eder** ve uygun patch'leri uygular. Mekanizma:

### Registry'ler

**`unsloth/models/mapper.py`** — model adından canonical model'a mapping:
```python
INT_TO_FLOAT_MAPPER = {
    'unsloth/llama-3-8b-bnb-4bit': 'meta-llama/Meta-Llama-3-8B',
    'unsloth/qwen3-1.7b-bnb-4bit': 'Qwen/Qwen3-1.7B',
    # ...
}
```
Bu mapping sayesinde 4-bit Unsloth versiyonu yüklediğinde, base model adını bilir ve doğru patch'leri uygular.

**`FORCE_FLOAT32`** — bazı model tiplerinde bf16 sayısal sorun çıkarır, bu liste onları FP32'ye zorlar.

**`DISABLE_SDPA_MODEL_NAMES`** — Scaled Dot-Product Attention'ın bug verdiği modeller; flash attention veya custom attention'a fallback.

### Tespit Akışı

```
FastModel.from_pretrained('Qwen/Qwen3-1.7B-Base')
  ↓
1. AutoConfig ile model_type oku → 'qwen3'
  ↓
2. Architecture'a bak → 'Qwen3ForCausalLM'
  ↓
3. Decision tree:
     - vision_config var mı? → text-only
     - 'MoE' suffix var mı? → standart text
     - INT_TO_FLOAT_MAPPER'da var mı? → mapping yap
  ↓
4. unsloth/models/qwen3.py yükle
  ↓
5. Qwen3-spesifik patch'ler aktif et:
     - Qwen3DecoderLayer.forward → fused
     - Qwen3RMSNorm.forward → Triton
     - vs.
  ↓
6. Model'i yükle (HF normal akışı)
  ↓
7. Post-load patches uygula (peft layer'ları için)
```

### `FastModel`'in Avantajı

Eski `FastLanguageModel` Llama-tarzı modelleri varsayıyordu. `FastModel` model_type'a göre route ediyor:
- `qwen3` → `qwen3.py`
- `llama` → `llama.py`
- `mistral` → `mistral.py`
- `qwen3_moe` → `qwen3_moe.py` (MoE-spesifik)

In [ ]:
# Hangi model'lerin spesifik patch'i var?
import os

models_dir = os.path.join(os.path.dirname(__import__('unsloth').__file__), 'models')
model_files = sorted([f for f in os.listdir(models_dir) if f.endswith('.py') and not f.startswith('_')])
print('Model-specific patches:')
for f in model_files:
    path = os.path.join(models_dir, f)
    with open(path) as fh:
        lines = sum(1 for _ in fh)
    print(f'  {f:30s} {lines:5d} lines')

## 6. Memory Tasarrufu — Fused Operasyonların Matematiği

**Unsloth'un %70 daha az VRAM iddiası nereden?** Üç ana kaynak:

### A. Activation Memory (en büyük katkı)

Forward'da her layer activation'ları VRAM'de tutar (backward için). Standart PyTorch:

```
Layer 1 input: x₁  ← saved
Layer 1 output: y₁ = f(x₁)  ← saved (backward için)
Layer 2 input: x₂ = y₁  ← saved
Layer 2 output: y₂ = g(x₂)  ← saved
...
```

Her layer için 2 activation. 32 layer modelde **64 activation tensor**.

**Unsloth gradient checkpointing (`use_gradient_checkpointing='unsloth'`)** Sadece checkpoint pozisyonlarındaki activation'ları tutar, geri kalanları **backward'da yeniden hesaplar**. 

Standart PyTorch gradient checkpointing de bunu yapar ama Unsloth'unki:
- CPU'ya offload da edebilir (async)
- Activation recompute'u Triton kernel'da yapar (daha hızlı recompute)

### B. Logits/Loss Memory (CE kernel sayesinde)

Standart cross-entropy:
```
logits: [batch, seq, vocab] = [4, 2048, 150000]  → 2.4 GB (bf16)
log_softmax(logits): aynı boyut → 2.4 GB daha
Total: ~5 GB sadece loss için
```

Fused CE kernel:
- `logits` tensor'u **hiç oluşturulmaz**
- log_softmax tile-by-tile hesaplanır, sadece target index'teki değer alınır
- VRAM: ~150 MB (binlerce kat az)

### C. LoRA Adapter Memory

Standart PEFT LoRA:
```
x → x_lora_in → A @ x_lora_in → B @ ... → x_lora_out
      (saved)         (saved)            (saved)
```

Unsloth fused LoRA: tek pass'te hesaplar, intermediate tensor saklanmaz.

### Sayısal Karşılaştırma

Llama-3-8B QLoRA, batch=2, seq=2048:

| Component | PyTorch + PEFT | Unsloth | Savings |
|-----------|----------------|---------|---------|
| Model weights (4-bit) | 4.5 GB | 4.5 GB | 0% |
| LoRA adapters | 0.05 GB | 0.05 GB | 0% |
| Activations | 8.0 GB | 2.5 GB | -69% |
| Logits (CE) | 5.0 GB | 0.15 GB | -97% |
| Optimizer state (8-bit AdamW) | 0.4 GB | 0.4 GB | 0% |
| Gradients | 0.5 GB | 0.5 GB | 0% |
| **Toplam (peak)** | **~18 GB** | **~8 GB** | **-56%** |

Bu rakamlar yaklaşık — gerçek değer model/seq/batch'e göre değişir.

## 7. `use_gradient_checkpointing='unsloth'` Detay

Standart options:
- `False` — checkpointing kapalı, en hızlı ama VRAM hungry
- `True` — PyTorch standart checkpointing
- `'unsloth'` — Unsloth özel, daha agresif

### Standart PyTorch Checkpointing

Forward pass'te belirli layer'larda activation'ları **fırlatır** (drop). Backward'da o layer'a gelince **forward'u yeniden çalıştırır**.

```
Forward:  layer 1 → save → layer 2 → drop → layer 3 → save → ...
Backward: ... layer 3 done → layer 2 grad lazım → forward'u yeniden çalıştır
```

Trade-off: %30-50 VRAM tasarrufu, %20-30 hız kaybı.

### Unsloth Checkpointing

Üç ek optimizasyon:

**1. CPU Offload (async)**
Activation'ları drop etmek yerine CPU RAM'e gönder. Backward'da geri al. Bilinen overlap pattern'ı:
```
Forward: layer 2 done
  └─→ async copy activation to CPU (overlap with layer 3 forward)
Backward: layer 2 grad lazım
  └─→ async copy from CPU (overlap with layer 3 backward)
```

PCIe bandwidth yeterse hız kaybı minimal.

**2. Triton Recompute Kernels**
Drop edilen layer'ları yeniden hesaplarken Triton kernel kullan. Standart PyTorch'tan ~%30 daha hızlı recompute.

**3. Smart Drop Selection**
Hangi layer'ları drop edeceğini cost-aware seçer:
- Pahalı layer'lar (dense FFN) → tutulur
- Ucuz layer'lar (norm, residual) → drop edilir

### Pratik

Hangi modelde ne fark eder?

| Model | Checkpoint = False | True | 'unsloth' |
|-------|--------------------|------|-----------|
| Llama-3-8B (LoRA) | 28 GB | 22 GB | **18 GB** |
| Qwen3-32B (QLoRA) | 60 GB | 38 GB | **24 GB** |
| Speed (relative) | 1.0× | 0.75× | **0.85×** |

Yani `'unsloth'` standart True'dan daha az VRAM **VE** daha hızlı.

## 8. Debug — Internals'i Bilince Hata Anlamak Kolaylaşıyor

### Hata 1: `ImportError: please install transformers via pip install`
Sebep: Unsloth `transformers` kütüphanesini patch etmeye çalışıyor ama bulamıyor.

### Hata 2: `RuntimeError: PASSING ZERO LENGTH TENSOR ...`
Sebep: Triton kernel boş tensor'la çağrıldı. Genelde batch=0 edge case.

### Hata 3: `KeyError: 'qwen3_5'`
Sebep: Model registry'de bilinmeyen model_type. transformers eski.

### Hata 4: `Unsloth: Llama doesn't have rotary embedding cached!`
Sebep: RoPE cache patch'lenmedi. `import unsloth` muhtemelen başta değil.

### Hata 5: `ZeroDivisionError: All labels in your dataset are -100`
Sebep: Fused CE kernel 0 token üzerinde loss hesaplayamıyor (bölme). Mask tüm sequence'ı maskelemiş.

### Pattern
İmport'a bak — yanlış mı? 
Model registry'de tanınıyor mu — `print(model.config.model_type)` 
Patch'ler aktif mi — `unsloth.__version__` ve `transformers.__version__` uyumlu mu 
Triton compile error mı — `TRITON_INTERPRET=1` ile re-run

## Özet

Bu notebook'tan sonra:

✅ Unsloth'un iki paket (`unsloth` + `unsloth_zoo`) ve içlerindeki katmanları biliyorsun 
✅ `import unsloth` neden başta — patch mekanizması net 
✅ Hangi Triton kernel'lar var ve neyi optimize ediyorlar 
✅ Manuel backprop neden gerekli, autograd'in tradeoff'u 
✅ Model registry — `model_type` → spesifik patch yolu 
✅ Memory tasarrufunun matematiği — activation, CE, LoRA 
✅ `use_gradient_checkpointing='unsloth'` ne yapıyor 

**Sıradaki notebook:** `02_quantization.ipynb` — `bnb-4bit` vs `unsloth-bnb-4bit` (Dynamic 2.0) detaylı, ne zaman hangi quant?